# Step D on Kaggle: evaluate all 15 models

Runs `evaluation/evaluate_v2.py`: 5 models (M0, M1, M4, M5, M6) x seeds 0, 1, 2 on full 3D volumes under the 13 DEC-005 conditions. The 15 checkpoints are in the repo (`results/train_v3/`).

**One-time setup**
1. Kaggle account with a **verified phone number** (needed for GPU and Internet).
2. **+ Add Input** -> search `brats20-dataset-training-validation` (by awsaf49) -> add it.
3. **Settings** (right panel): Accelerator **GPU T4 x2** (one GPU is used), **Internet on**.
4. Set `REPO` below to your fork.

**Order matters (see `README.md`)**
1. `SPLIT = 'val'`, `MAX_PATIENTS = 2`: smoke test. Then `MAX_PATIENTS = 0`: all 37 validation patients and the acceptance checks.
2. Only when every check passes: `SPLIT = 'test'`, `CONFIRM_TEST = True`. **The test set is run once**; nobody looks at test numbers before it finishes.

Run it with **Save Version -> Save & Run All (Commit)**: it keeps running with the browser closed (Kaggle limit 12 h; no new patient starts after `TIME_LIMIT_H`). If it stops early, open the notebook, **+ Add Input -> this notebook's own output**, and commit again: finished (patient, condition) pairs are restored and skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
# ---- Configuration ----
import glob, os, shutil, time
T0 = time.time()
SPLIT        = 'val'            # 'val' while debugging; 'test' for the one final run
CONFIRM_TEST = False            # must be True for SPLIT = 'test'
MAX_PATIENTS = 2                # 2 = smoke test; 0 = all patients of the split
TIME_LIMIT_H = 10.0             # do not START a new patient after this (Kaggle stops at 12 h)
REPO, BRANCH = 'https://github.com/muhammadmapkar/Argus.git', 'nazir-step-d'
RAW = glob.glob('/kaggle/input/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]   # BraTS, already extracted
OUT = f'/kaggle/working/eval_v2/{SPLIT}'
os.makedirs(OUT, exist_ok=True)
print('BraTS at', RAW, '|', len([d for d in os.listdir(RAW) if d.startswith('BraTS20')]), 'patients')

In [ ]:
!git clone -q --depth 1 -b {BRANCH} {REPO} /kaggle/temp_repo 2>/dev/null || git -C /kaggle/temp_repo pull -q
%cd /kaggle/temp_repo
!pip -q install nibabel medpy
!ls results/train_v3/*/seed_*/best_model.pt | wc -l      # expect 15

In [ ]:
# Resume: restore an earlier run of this notebook (added as an input), if any
prev = glob.glob(f'/kaggle/input/**/eval_v2/{SPLIT}/raw_metrics_v2.csv', recursive=True)
if prev and not os.path.exists(f'{OUT}/raw_metrics_v2.csv'):
    src = os.path.dirname(prev[0])
    shutil.copytree(src, OUT, dirs_exist_ok=True)
    print('Restored earlier results from', src)
else:
    print('Starting fresh' if not prev else 'Already restored')

In [ ]:
!python -m pytest tests/test_evaluate_v2.py -q

In [ ]:
# Evaluation. Prints seconds per (patient, condition) and the projected time of the full test run.
test_flag = '--confirm_test' if CONFIRM_TEST else ''
!python -m evaluation.evaluate_v2 --raw_dir {RAW} --split {SPLIT} {test_flag} --max_patients {MAX_PATIENTS} \
    --time_limit_h {TIME_LIMIT_H} --out_csv {OUT}/raw_metrics_v2.csv --out_dir {OUT}

## Acceptance checks (validation only)
Dice reproduces the training logs within 0.005 (needs all 37 validation patients, so `MAX_PATIENTS = 0`), HD95 empty-mask convention, NIfTI round trip.

In [ ]:
if SPLIT == 'val':
    !python -m evaluation.check_val --raw_dir {RAW} --csv {OUT}/raw_metrics_v2.csv --pred_dir {OUT}/predictions
else:
    print('Acceptance checks are for the validation run only.')

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUT}/raw_metrics_v2.csv')
print(len(df), 'rows,', df.patient_id.nunique(), 'patients,', df.groupby(['model', 'seed']).ngroups, 'checkpoints')
df[df.condition_id.isin(['clean', 'kspace_snr5_r1.0_t1', 'kspace_snr3_r0.5_t5'])] \
  .groupby(['model', 'condition_id']).dice_mean.mean().unstack().round(4)

In [ ]:
# Test run only: zip the seed-0 predictions for Joshua's MATLAB pipeline (not for git)
if SPLIT == 'test':
    !cd {OUT} && zip -qr predictions_test_seed0.zip predictions && ls -lh predictions_test_seed0.zip
    print('After the run: Output tab -> download raw_metrics_v2.csv (commit it) and predictions_test_seed0.zip (send to Joshua).')